#### Using the same idea as the interpolation of resident numbers

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [3]:
# get subzone data with their corresponding areas in km^2
def get_subzone_data(year: int):
    """
    Args
    ------
    year: int
        year of the subzones you want (2019, 2014, 2008)

    Returns
    ------
    Subzone dataframe containing the subzone names, planning area names and areas of points of interest.
    """

    year = str(year)

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_boundary_{year}.gpkg")
    subzones_gpd = gpd.read_file(subzones_filepath)
    subzones_gpd = subzones_gpd.map(lambda s: s.lower() if isinstance(s, str) else s)
    # lower() the column names of subzones_gpd
    subzones_gpd.columns = subzones_gpd.columns.str.lower()    

    # there is no landuse data found for 2008 masterplan, 
    # so not able to merge with the subzone_classification file obtained from extracting_landuse_type.ipynb
    if year == "2008":
        return subzones_gpd

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    subzones_csv = pd.read_csv(subzones_filepath)
    subzones_csv = subzones_csv.map(lambda s: s.lower() if isinstance(s, str) else s)

    subzones_with_geom = subzones_csv.merge(
        subzones_gpd[["subzone_n", "geometry"]],
        on = "subzone_n",
        how = "left"
    )

    missing_count = subzones_with_geom[subzones_with_geom.columns[-1]].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} subzones did not find a match in the CSV.")

    return subzones_with_geom

In [4]:
def get_ethnicity_data(year: int):

    year = str(year)

    ethnicity_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ethnicity_combined.xlsx")
    ethnicity_df = pd.read_excel(ethnicity_filepath, sheet_name = f"{year}")

    ethnicity_df = ethnicity_df.rename(columns = {
        "subzone": "subzone_n",
        "planning_area": "pln_area_n"
    })

    ethnicity_df["pln_area_n"] = ethnicity_df["pln_area_n"].ffill()

    ## the subzones for punggol is named differently in the masterplan for 2010 and 2015, 
    # instead of eg: subzone 2, it is sz2, so renaming it
    if year == "2015" or year == "2010":
        ethnicity_df.loc[
            (ethnicity_df["subzone_n"] == "subzone 2") &
            (ethnicity_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz2"
        ethnicity_df.loc[
            (ethnicity_df["subzone_n"] == "subzone 3") &
            (ethnicity_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz3"
        ethnicity_df.loc[
            (ethnicity_df["subzone_n"] == "subzone 4") &
            (ethnicity_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz4"
        ethnicity_df.loc[
            (ethnicity_df["subzone_n"] == "subzone 5") &
            (ethnicity_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz5"

    return ethnicity_df

In [5]:
def convert_to_geodataframe(df):
    if not isinstance(df, gpd.GeoDataFrame):
        df = gpd.GeoDataFrame(df, geometry='geometry')

    # ensure the original data has the correct starting CRS (WGS84)
    # (Only needed if it isn't already set; usually .gpkg files have this metadata)
    if df.crs is None:
        df.set_crs(epsg=4326, inplace=True)

    # transform the entire GeoDataFrame to SVY21 (Meters)
    df_meters = df.to_crs(epsg=3414)

    return df_meters

In [6]:
subzone_df = get_subzone_data(2008)
subzone_df = convert_to_geodataframe(subzone_df)
subzone_df['area_sqm'] = subzone_df.geometry.area
subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000
subzone_areas = subzone_df[['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', "geometry"]].copy()

ethnicity_df = get_ethnicity_data(2010)


In [7]:
def calculate_density(ethnicity_df, subzone_areas):

    ethnicity_columns = ['pln_area_n', 'subzone_n',
       'male_chinese', 'female_chinese',
       'male_malays', 'female_malays',
       'male_indians', 'female_indians',
       'male_others', 'female_others']
        
    ethnicity_df = ethnicity_df[ethnicity_columns].copy()

    # standardize casing and remove hidden whitespace
    subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
    ethnicity_df['subzone_n'] = ethnicity_df['subzone_n'].astype(str).str.strip().str.lower()

    density_df = subzone_areas.merge(
        ethnicity_df,
        on = ["subzone_n", "pln_area_n"],
        how = "left"
    )
    
    for ethinicty_gender in ethnicity_columns[2:]:
        new_column_name = "density_" + ethinicty_gender

        density_df[f"density_{ethinicty_gender}"] =  density_df[ethinicty_gender] / density_df["area_sqkm"]

    return density_df
    

In [8]:
density_df = calculate_density(ethnicity_df, subzone_areas)
density_df.head()

,subzone_n,pln_area_n,area_sqm,area_sqkm,geometry,male_chinese,female_chinese,male_malays,female_malays,male_indians,...,male_others,female_others,density_male_chinese,density_female_chinese,density_male_malays,density_female_malays,density_male_indians,density_female_indians,density_male_others,density_female_others
0,woodlands regional centre,woodlands,5.956521e+05,0.595652,"MULTIPOLYGON (((22629.486 46980.246, 22683.012...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,kranji,sungei kadut,3.654121e+06,3.654121,"MULTIPOLYGON (((20329.947 47234.988, 20333.366...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,turf club,sungei kadut,3.290816e+06,3.290816,"MULTIPOLYGON (((21094.367 44815.121, 21099.854...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,nee soon,yishun,2.209311e+06,2.209311,"MULTIPOLYGON (((26860.555 43913.328, 26839.719...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,yishun west,yishun,1.517705e+06,1.517705,"MULTIPOLYGON (((28228.609 45792.488, 28223.039...",20417.0,20694.0,5457.0,5408.0,3385.0,...,793.0,840.0,13452.551397,13635.063849,3595.561198,3563.275601,2230.341699,2124.919344,522.499547,553.467364


In [9]:
def sanity_checks(density_df, subzone_areas, ethnicity_df):
    duplicate_mask = density_df.duplicated(subset=['subzone_n'], keep=False)

    # Get the unique names of those duplicates
    duplicate_names = density_df.loc[duplicate_mask, 'subzone_n'].unique()

    print("Planning areas with duplicates:")
    print(list(duplicate_names))

    ## checking if merging was successful. 
    # this checks which subzone and planning area pair from the ethnicity_df did not match with subzone_areas dataframe.
    subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
    # subzone_areas.to_csv("subzone_areas.csv")
    ethnicity_df['subzone_n'] = ethnicity_df['subzone_n'].astype(str).str.strip().str.lower()
    # ethnicity_df.to_csv("ethnicity_df.csv")

    check_df = subzone_areas[['subzone_n', 'pln_area_n']].merge(
        ethnicity_df[['subzone_n', 'pln_area_n']], 
        on=['subzone_n', 'pln_area_n'],
        how='outer',
        indicator=True
    )

    failed_df = check_df[check_df['_merge'] == 'right_only'][['subzone_n', 'pln_area_n']]
    print(failed_df.to_string(index=False))

In [10]:
sanity_checks(density_df, subzone_areas, ethnicity_df)

Planning areas with duplicates:
['seletar', 'boon lay', 'hong kah', 'central', 'pang sua', 'trafalgar', 'marymount', 'tanjong pagar', 'marina east']
subzone_n      pln_area_n
    total      ang mo kio
    total           bedok
    total          bishan
    total     bukit batok
    total     bukit merah
    total   bukit panjang
    total     bukit timah
    total          changi
    total   choa chu kang
    total        clementi
    total   downtown core
    total         geylang
    total         hougang
    total     jurong east
    total     jurong west
    total         kallang
    total          mandai
    total   marine parade
    total          newton
    total          novena
    total          others
    total          outram
    total       pasir ris
    total         punggol
    total      queenstown
    total    river valley
    total          rochor
    total       sembawang
    total        sengkang
    total       serangoon
    total singapore river
    total        ta

In [11]:
ethnicity_annual_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/ResidentsByAgeGroupEthnicGroupAndSexAnnual.csv")
ethnicity_annual_df = pd.read_csv(ethnicity_annual_filepath)

# .lower() all the str in the dataframe
ethnicity_annual_df = ethnicity_annual_df.map(lambda s: s.lower() if isinstance(s, str) else s)
# lower() the column names of ethnicity_annual_df
ethnicity_annual_df.columns = ethnicity_annual_df.columns.str.lower()
ethnicity_annual_df.head()

,dataseries,2025,2024,2023,2022,2021,2020,2019,2018,2017,...,1966,1965,1964,1963,1962,1961,1960,1959,1958,1957
0,total residents,4204515,4180868,4149253,4073239,3986842,4044210,4026209,3994283,3965796,...,1934400,1886900,1841600,1795000,1750200,1702400,1646400,1587200,1518800,1445929
1,0 - 4 years,165550,170496,175268,178085,178435,183076,185355,185528,187653,...,289600,293500,299900,303000,306100,303700,297000,288800,277800,264727
2,5 - 9 years,199813,202082,201974,201360,198760,198737,197775,199066,200575,...,297600,291100,284000,276000,266700,258900,250300,241000,229600,218097
3,10 - 14 years,206577,204391,204459,202379,199993,206393,207926,206530,206253,...,257300,248500,240200,232000,223900,217400,200300,175700,157100,136280
4,15 - 19 years,211394,211067,209579,206749,204913,215234,222222,226520,232973,...,215200,198600,175800,159000,140000,124900,124700,131500,134300,135679


In [12]:
density_df.columns
# # the word male captures both male and female
# ethnic_cols = [col for col in density_df.columns if "male" in col.lower()]
# gender_ethnic_cols = [col for col in ethnic_cols if not "density" in col]
# gender_ethnic_cols

Index(['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', 'geometry',
       'male_chinese', 'female_chinese', 'male_malays', 'female_malays',
       'male_indians', 'female_indians', 'male_others', 'female_others',
       'density_male_chinese', 'density_female_chinese', 'density_male_malays',
       'density_female_malays', 'density_male_indians',
       'density_female_indians', 'density_male_others',
       'density_female_others'],
      dtype='object')

#### Calculate Density ratio of ethnicity per subzone

In [13]:
def calculate_national_average_density(density_df):
    """
    return the density of each age group across whole of Singapore
    """

    # the word male captures both male and female
    ethnic_cols = [col for col in density_df.columns if "male" in col.lower()]
    gender_ethnic_cols = [col for col in ethnic_cols if not "density" in col]
    gender_ethnic_cols

    total_area_all_subzones = density_df['area_sqkm'].sum()

    aggregated_results = []

    for col in gender_ethnic_cols:
        total_resident_count = density_df[col].sum()
        avg_density = total_resident_count / total_area_all_subzones
        
        aggregated_results.append({
            'ethnic_group': col,
            'national_avg_density': avg_density
        })

    national_density_df = pd.DataFrame(aggregated_results)

    return national_density_df

In [14]:
national_density_df = calculate_national_average_density(density_df)
national_density_df

,ethnic_group,national_avg_density
0,male_chinese,1756.644954
1,female_chinese,1825.189145
2,male_malays,322.722836
3,female_malays,325.404961
4,male_indians,230.702649
5,female_indians,214.994900
6,male_others,75.583642
7,female_others,83.559197


#### Calculate ethnicity Density Ratio of each subzone
$Ratio = \frac{Subzone\ Density}{National\ Average\ Density}$

In [15]:
def calculate_density_ratio(density_df, national_density_df):
    ethnic_group_list = national_density_df["ethnic_group"].tolist()
    
    for ethnic_group in ethnic_group_list:
        # get the national average density for the ethnic group
        national_average_density = national_density_df.loc[
            national_density_df["ethnic_group"] == ethnic_group,
            "national_avg_density"
        ].item()

        # if that age group is 0, set to np.nan to indicate that the national average was 0. 
        if national_average_density <= 0:
            density_df[f"density_ratio_{ethnic_group}"] = np.nan

        else:
            # calculate the density ratio for that ethnic group per subzone
            density_df[f"density_ratio_{ethnic_group}"] = density_df[f"density_{ethnic_group}"] / national_average_density

    return density_df

In [16]:
ethnic_density_ratio = calculate_density_ratio(density_df, national_density_df)

save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ethnic_density_ratios/ethnic_density_ratio_2010.csv")
ethnic_density_ratio.to_csv(save_to_file)
ethnic_density_ratio

,subzone_n,pln_area_n,area_sqm,area_sqkm,geometry,male_chinese,female_chinese,male_malays,female_malays,male_indians,...,density_male_others,density_female_others,density_ratio_male_chinese,density_ratio_female_chinese,density_ratio_male_malays,density_ratio_female_malays,density_ratio_male_indians,density_ratio_female_indians,density_ratio_male_others,density_ratio_female_others
0,woodlands regional centre,woodlands,5.956521e+05,0.595652,"MULTIPOLYGON (((22629.486 46980.246, 22683.012...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,kranji,sungei kadut,3.654121e+06,3.654121,"MULTIPOLYGON (((20329.947 47234.988, 20333.366...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,turf club,sungei kadut,3.290816e+06,3.290816,"MULTIPOLYGON (((21094.367 44815.121, 21099.854...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,nee soon,yishun,2.209311e+06,2.209311,"MULTIPOLYGON (((26860.555 43913.328, 26839.719...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,yishun west,yishun,1.517705e+06,1.517705,"MULTIPOLYGON (((28228.609 45792.488, 28223.039...",20417.0,20694.0,5457.0,5408.0,3385.0,...,522.499547,553.467364,7.658094,7.470494,11.141329,10.950281,9.667603,9.883580,6.912865,6.623656
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,sungei serangoon west,sengkang,1.572812e+06,1.572812,"MULTIPOLYGON (((37009.969 40867.361, 37011.087...",21915.0,22183.0,3199.0,3269.0,2397.0,...,371.945391,410.729441,7.931965,7.727441,6.302428,6.387252,6.606003,7.035413,4.920977,4.915431
307,sungei serangoon,hougang,1.333531e+06,1.333531,"MULTIPOLYGON (((36538.391 39761.352, 36517.439...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,hougang central,hougang,2.291220e+06,2.291220,"MULTIPOLYGON (((35948.803 40264.817, 35951.927...",20669.0,20927.0,1805.0,1840.0,2031.0,...,202.075724,251.830870,5.135333,5.004172,2.441072,2.467896,3.842293,4.068204,2.673538,3.013802
309,changi west,changi,4.849019e+06,4.849019,"MULTIPOLYGON (((45347.86 41116.219, 45350.666 ...",305.0,338.0,186.0,147.0,199.0,...,4.537000,4.537000,0.035807,0.038190,0.118858,0.093162,0.177888,0.152516,0.060026,0.054297


In [17]:
mapping_data = [
    ("jurong east", "boon lay", "yuhua west"),
    ("jurong west", "boon lay", "boon lay place"),
    ("sengkang", "buangkok", "anchorvale"),
    ("bukit panjang", "bukit panjang", "jelebu"),
    ("choa chu kang", "central", "choa chu kang central"),
    ("jurong west", "central", "jurong west central"),
    ("pasir ris", "elias", "pasir ris west"),
    ("bukit batok", "hong kah", "hong kah north"),
    ("sengkang", "jalan kayu east", "fernvale"),
    ("sengkang", "jalan kayu west", "sengkang west"),
    ("jurong east", "jurong lake", "lakeside"),
    ("jurong east", "jurong regional centre", "jurong gateway"),
    ("toa payoh", "kallang", "lorong 8 toa payoh"),
    ("choa chu kang", "kranji north", "choa chu kang north"),
    ("toa payoh", "marymount", "toa payoh west"),
    ("choa chu kang", "pang sua", "choa chu kang north"),
    ("pasir ris", "pasir ris town", "pasir ris central"),
    ("toa payoh", "paya lebar", "joo seng"),
    ("hougang", "rosyth", "kovan"),
    ("ang mo kio", "sindo", "tagore"),
    ("punggol", "subzone 2", "punggol town centre"),
    ("punggol", "subzone 3", "matilda"),
    ("punggol", "subzone 4", "punggol field"),
    ("punggol", "subzone 5", "waterway east"),
    ("sengkang", "sungei serangoon west", "rivervale"),
    ("hougang", "tai keng", "tai seng"),
    ("ang mo kio", "town centre", "ang mo kio town centre"),
    ("sengkang", "trafalgar", "compassvale"),
    ("jurong east", "yuhua", "yuhua east")
]

In [18]:
def perform_step_1_and_2():
    
    # dictionary of the available demographic data years and their corresponding subzone data years
    anchor_years = {2010: 2008,
                    2015: 2014,
                    2020: 2019}

    for ethnicity_year, subzone_year in anchor_years.items():
        subzone_df = get_subzone_data(subzone_year)
        subzone_df_meters = convert_to_geodataframe(subzone_df)

        # calculate area
        subzone_df['area_sqm'] = subzone_df_meters.geometry.area
        subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000
        subzone_areas = subzone_df[['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', "geometry"]].copy()

        ethnicity_df = get_ethnicity_data(ethnicity_year)
        density_df = calculate_density(ethnicity_df, subzone_areas)
        # fill NaN results with 0 (0 population data for that subzone)
        density_df = density_df.fillna(np.nan)

        print(f"\nPerforming sanity check for {ethnicity_year}")
        sanity_checks(density_df, subzone_areas, ethnicity_df)

        national_age_grp_density_df = calculate_national_average_density(density_df)
        density_ratio_df = calculate_density_ratio(density_df, national_age_grp_density_df)

        if subzone_year == 2008:
            
            map_df = pd.DataFrame(mapping_data, columns=['pln_area_n', 'subzone_n', 'new_subzone'])
            # renaming certain subzone names so they would match with subzone names from 2014
            # Merge this with your density_ratio_df to update the names

            map_df['pln_area_n'] = map_df['pln_area_n'].str.strip().str.lower()
            map_df['subzone_n'] = map_df['subzone_n'].str.strip().str.lower()
            map_df['new_subzone'] = map_df['new_subzone'].str.strip().str.lower()

            density_ratio_df['pln_area_n'] = density_ratio_df['pln_area_n'].str.strip().str.lower()
            density_ratio_df['subzone_n'] = density_ratio_df['subzone_n'].str.strip().str.lower()
            density_ratio_df = density_ratio_df.merge(
                map_df, 
                on=['pln_area_n', 'subzone_n'], 
                how='left'
            )

            # Replace the old subzone with the new one where a match was found
            density_ratio_df['subzone_n'] = density_ratio_df['new_subzone'].fillna(density_ratio_df['subzone_n'])
            density_ratio_df.drop(columns=['new_subzone'], inplace=True)
        
            # COMBINE DUPLICATES Group by the identifiers and sum all numeric columns (density, ratios, etc.)
            # We use reset_index() to keep the dataframe structure flat
            density_ratio_df = density_ratio_df.groupby(['pln_area_n', 'subzone_n']).sum(numeric_only=True).reset_index()
        
        save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ethnic_density_ratios/ethnicity_density_ratio_{ethnicity_year}.csv")
        density_ratio_df.to_csv(save_to_file)

In [19]:
perform_step_1_and_2()


Performing sanity check for 2010
Planning areas with duplicates:
['seletar', 'boon lay', 'hong kah', 'central', 'pang sua', 'trafalgar', 'marymount', 'tanjong pagar', 'marina east']
subzone_n      pln_area_n
    total      ang mo kio
    total           bedok
    total          bishan
    total     bukit batok
    total     bukit merah
    total   bukit panjang
    total     bukit timah
    total          changi
    total   choa chu kang
    total        clementi
    total   downtown core
    total         geylang
    total         hougang
    total     jurong east
    total     jurong west
    total         kallang
    total          mandai
    total   marine parade
    total          newton
    total          novena
    total          others
    total          outram
    total       pasir ris
    total         punggol
    total      queenstown
    total    river valley
    total          rochor
    total       sembawang
    total        sengkang
    total       serangoon
    total s

#### Temporal Interpolation of Weights

In [20]:
read_from_file_2010 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ethnic_density_ratios/ethnicity_density_ratio_2010.csv")
read_from_file_2015 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ethnic_density_ratios/ethnicity_density_ratio_2015.csv")
read_from_file_2020 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ethnic_density_ratios/ethnicity_density_ratio_2020.csv")

density_ratio_df_2010 = pd.read_csv(read_from_file_2010)

density_ratio_df_2015 = pd.read_csv(read_from_file_2015)

density_ratio_df_2020 = pd.read_csv(read_from_file_2020)


In [21]:
# vertical concatenation of 2010, 2015, 2020 dataframes
base_ids = ['subzone_n', 'pln_area_n']

# add Year tags and filter for your specific columns
# 2010 Data
cols_2010 = base_ids + [col for col in density_ratio_df_2010.columns if "density_ratio_" in col.lower()]
d10 = density_ratio_df_2010[cols_2010].copy()
d10['Year'] = 2010

# 2015 Data
cols_2015 = base_ids + [col for col in density_ratio_df_2015.columns if "density_ratio_" in col.lower()]
d15 = density_ratio_df_2015[cols_2015].copy()
d15['Year'] = 2015

# 2020 Data
cols_2020 = base_ids + [col for col in density_ratio_df_2020.columns if "density_ratio_" in col.lower()]
d20 = density_ratio_df_2020[cols_2020].copy()
d20['Year'] = 2020

# pd.concat will automatically handle the differing columns by adding NaNs where data is missing
df_stacked = pd.concat([d10, d15, d20], axis=0, ignore_index=True, sort=False)

# Melt into Long Format
# This will create a single "Ethnicity" column and a "WeightingRatio" column
df_long = df_stacked.melt(
    id_vars=['subzone_n', 'pln_area_n', 'Year'],
    var_name='Ethnicity_Ratio',
    value_name='WeightingRatio'
)

# remove 'density_ratio_' prefix)
df_long['Ethnicity'] = df_long['Ethnicity_Ratio'].str.replace('density_ratio_', '')
df_long = df_long.drop(columns = ["Ethnicity_Ratio"])

# save_to_file = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/consolidated_data.csv")
# df_long.to_csv(save_to_file)

display(df_long)


,subzone_n,pln_area_n,Year,WeightingRatio,Ethnicity
0,ang mo kio town centre,ang mo kio,2010,3.772573,male_chinese
1,cheng san,ang mo kio,2010,7.195530,male_chinese
2,chong boon,ang mo kio,2010,6.355962,male_chinese
3,kebun bahru,ang mo kio,2010,5.366168,male_chinese
4,seletar,ang mo kio,2010,0.000000,male_chinese
...,...,...,...,...,...
7707,springleaf,yishun,2020,0.302884,female_others
7708,yishun central,yishun,2020,0.766709,female_others
7709,yishun east,yishun,2020,5.988223,female_others
7710,yishun south,yishun,2020,5.575427,female_others


In [22]:
def interpolate_ethnicity_ratios(df_long):
    # 1. Setup identifiers
    subzones = df_long[['subzone_n', 'pln_area_n']].drop_duplicates()
    years = np.arange(2010, 2021)
    ethnicities = df_long['Ethnicity'].unique()
    
    # 2. Create the Backbone (Every Subzone x Every Year x Every Ethnicity)
    df_backbone = subzones.assign(k=1).merge(pd.DataFrame({'Year': years, 'k': 1}), on='k')
    df_backbone = df_backbone.assign(k=1).merge(pd.DataFrame({'Ethnicity': ethnicities, 'k': 1}), on='k').drop('k', axis=1)
    
    # 3. Merge with Anchor Data
    df_interp = pd.merge(
        df_backbone, 
        df_long, 
        on=['subzone_n', 'pln_area_n', 'Year', 'Ethnicity'], 
        how='left'
    )
    
    # 4. Apply "Decay to Zero" Logic
    # Force 0s only at the anchor years (2010, 2015, 2020) if a subzone is missing.
    # This creates the target for the linear trend.
    anchor_years = [2010, 2015, 2020]
    mask = df_interp['Year'].isin(anchor_years)
    df_interp.loc[mask, 'WeightingRatio'] = df_interp.loc[mask, 'WeightingRatio'].fillna(0)
    
    # 5. Perform Linear Interpolation
    # Sort to ensure years are in order for each group before interpolating
    df_interp = df_interp.sort_values(['subzone_n', 'pln_area_n', 'Ethnicity', 'Year'])
    
    df_interp['WeightingRatio'] = df_interp.groupby(['subzone_n', 'pln_area_n', 'Ethnicity'])['WeightingRatio'].transform(
        lambda x: x.interpolate(method='linear')
    )
    
    return df_interp

# Run the simplified interpolation
df_ethnicity_interpolated = interpolate_ethnicity_ratios(df_long)

# Save result
save_path = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/ethnic_density_ratios/ethnicity_interpolation.csv")
df_ethnicity_interpolated.to_csv(save_path, index=False)

In [23]:
# granular_output_dir = Path("sanity_check/granular_ethnicity_checks")
# granular_output_dir.mkdir(parents=True, exist_ok=True)

# def plot_granular_trends(df, subzone_name, pln_area):
#     """
#     Plots annual trends for all granular Ethnicitys in a subzone.
#     """
#     # Filter for the specific subzone
#     subset = df[(df['subzone_n'] == subzone_name) & (df['pln_area_n'] == pln_area)]
    
#     if subset.empty:
#         return

#     # Create the plot
#     plt.figure(figsize=(14, 8))
    
#     # Use a color palette to distinguish Ethnicitys
#     palette = sns.color_palette("husl", len(subset['Ethnicity'].unique()))
    
#     # Plot interpolated lines for all groups
#     sns.lineplot(
#         data=subset, 
#         x='Year', 
#         y='WeightingRatio', 
#         hue='Ethnicity', 
#         marker='o', 
#         palette=palette,
#         alpha=0.7
#     )
    
#     # Highlight Anchor Years (2010, 2015, 2020) with larger dots
#     # This helps verify if the interpolation is correctly hitting the data points
#     anchors = subset[subset['Year'].isin([2010, 2015, 2020])]
#     plt.scatter(
#         anchors['Year'], 
#         anchors['WeightingRatio'], 
#         color='black', 
#         s=40, 
#         zorder=5, 
#         label='Anchor Years'
#     )
    
#     plt.title(f"Granular Ethnicity Trends: {subzone_name} ({pln_area})")
#     plt.xlabel("Year")
#     plt.ylabel("Weighting Ratio")
#     plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Ethnicity')
#     plt.grid(True, linestyle='--', alpha=0.5)
#     plt.tight_layout()
    
#     # Save the file
#     safe_name = subzone_name.replace("/", "_")
#     plt.savefig(granular_output_dir / f"{safe_name}_granular_trend.png")
#     plt.close()

# # 2. Run for sample subzones
# # We use head(20) to avoid generating too many files at once
# sample_list = df_ethnicity_interpolated[['subzone_n', 'pln_area_n']].drop_duplicates().head(100)

# for _, row in sample_list.iterrows():
#     plot_granular_trends(df_ethnicity_interpolated, row['subzone_n'], row['pln_area_n'])

# print(f"Granular trend plots saved to {granular_output_dir}")

#### Extrapolate for 2021

In [24]:
def extrapolate_ethnicity_to_2021(df_interp):
    # 1. Isolate the last two years to establish the trajectory
    df_2019 = df_interp[df_interp['Year'] == 2019].copy()
    df_2020 = df_interp[df_interp['Year'] == 2020].copy()
    
    # Rename columns to avoid confusion during calculation
    df_2019 = df_2019.rename(columns={'WeightingRatio': 'Ratio_2019'})
    df_2020 = df_2020.rename(columns={'WeightingRatio': 'Ratio_2020'})
    
    # 2. Merge data to calculate the specific trend for each subzone and ethnicity
    # We join on subzone and Ethnicity to ensure we are comparing the correct categories
    trend_df = pd.merge(
        df_2020[['subzone_n', 'pln_area_n', 'Ethnicity', 'Ratio_2020']],
        df_2019[['subzone_n', 'pln_area_n', 'Ethnicity', 'Ratio_2019']],
        on=['subzone_n', 'pln_area_n', 'Ethnicity'],
        how='left'
    )
    
    # Calculate the slope: Annual Change = 2020 value - 2019 value
    trend_df['Annual_Change'] = trend_df['Ratio_2020'] - trend_df['Ratio_2019']
    
    # 3. Project 2021 Data
    df_2021 = trend_df.copy()
    df_2021['WeightingRatio'] = df_2021['Ratio_2020'] + df_2021['Annual_Change']
    
    # 4. Apply Safety Constraints
    # Weighting ratios cannot be negative; clip any downward trends at 0
    df_2021['WeightingRatio'] = df_2021['WeightingRatio'].clip(lower=0)
    
    # Final formatting to match the original schema
    df_2021['Year'] = 2021
    df_2021 = df_2021[['subzone_n', 'pln_area_n', 'Year', 'Ethnicity', 'WeightingRatio']]
    
    # 5. Combine with the 2010-2020 master dataframe
    return pd.concat([df_interp, df_2021], ignore_index=True)

# Generate the 2010-2021 Master Ethnicity Ratio Dataframe
df_ethnicity_density_ratio = extrapolate_ethnicity_to_2021(df_ethnicity_interpolated)

# Save the final results
save_path = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/ethnic_density_ratios/ethnicity_master_2010_2021.csv")
df_ethnicity_density_ratio.to_csv(save_path, index=False)

#### Check that the final density ratio dataframe contains all the unique subzone and planning area pair

In [25]:
# Extract pairs from each anchor year
pairs_2010 = density_ratio_df_2010[['subzone_n', 'pln_area_n']]
pairs_2015 = density_ratio_df_2015[['subzone_n', 'pln_area_n']]
pairs_2020 = density_ratio_df_2020[['subzone_n', 'pln_area_n']]

# Combine to get the Master List of all unique subzones across the decade
master_subzones = pd.concat([pairs_2010, pairs_2015, pairs_2020]).drop_duplicates()
master_subzones

,subzone_n,pln_area_n
0,ang mo kio town centre,ang mo kio
1,cheng san,ang mo kio
2,chong boon,ang mo kio
3,kebun bahru,ang mo kio
4,seletar,ang mo kio
...,...,...
288,plantation,tengah
289,tengah industrial estate,tengah
311,bahar,western water catchment
312,cleantech,western water catchment


In [26]:
def check_interpolation_completeness(df_ethnicity_interpolated, master_subzones):
    years = sorted(df_ethnicity_interpolated['Year'].unique())
    total_master_count = len(master_subzones)
    
    print(f"Checking {total_master_count} master subzones across years {years[0]}-{years[-1]}...\n")
    
    report = []
    
    for year in years:
        # 1. Get unique pairs for the current year in the interpolated data
        current_year_pairs = df_ethnicity_interpolated[df_ethnicity_interpolated['Year'] == year][['subzone_n', 'pln_area_n']].drop_duplicates()
        
        # 2. Perform a 'left merge' to find what is in master but NOT in the year data
        check_merge = pd.merge(
            master_subzones, 
            current_year_pairs, 
            on=['subzone_n', 'pln_area_n'], 
            how='left', 
            indicator=True
        )
        
        missing = check_merge[check_merge['_merge'] == 'left_only']
        num_missing = len(missing)
        
        report.append({
            'Year': year,
            'Subzones_Present': len(current_year_pairs),
            'Missing_Count': num_missing
        })
        
        if num_missing > 0:
            print(f"Year {year}: Missing {num_missing} subzones!")
            # Uncomment the line below to see exactly which ones are missing
            # print(missing[['subzone_n', 'pln_area_n']])
        else:
            print(f"Year {year}: All {total_master_count} subzones present.")

    return pd.DataFrame(report)

# Run the check
completeness_report = check_interpolation_completeness(df_ethnicity_density_ratio, master_subzones)

Checking 356 master subzones across years 2010-2021...

Year 2010: All 356 subzones present.
Year 2011: All 356 subzones present.
Year 2012: All 356 subzones present.
Year 2013: All 356 subzones present.
Year 2014: All 356 subzones present.
Year 2015: All 356 subzones present.
Year 2016: All 356 subzones present.
Year 2017: All 356 subzones present.
Year 2018: All 356 subzones present.
Year 2019: All 356 subzones present.
Year 2020: All 356 subzones present.
Year 2021: All 356 subzones present.


#### Interpolation Step

In [27]:
read_from_path = Path(BASE_DATASET_PATH / "singapore_data/data_gov/ResidentsByAgeGroupEthnicGroupAndSexAnnual.csv")
df_national = pd.read_csv(read_from_path)
df_national.columns = df_national.columns.str.lower().str.strip()
df_national


,dataseries,2025,2024,2023,2022,2021,2020,2019,2018,2017,...,1966,1965,1964,1963,1962,1961,1960,1959,1958,1957
0,Total Residents,4204515,4180868,4149253,4073239,3986842,4044210,4026209,3994283,3965796,...,1934400,1886900,1841600,1795000,1750200,1702400,1646400,1587200,1518800,1445929
1,0 - 4 Years,165550,170496,175268,178085,178435,183076,185355,185528,187653,...,289600,293500,299900,303000,306100,303700,297000,288800,277800,264727
2,5 - 9 Years,199813,202082,201974,201360,198760,198737,197775,199066,200575,...,297600,291100,284000,276000,266700,258900,250300,241000,229600,218097
3,10 - 14 Years,206577,204391,204459,202379,199993,206393,207926,206530,206253,...,257300,248500,240200,232000,223900,217400,200300,175700,157100,136280
4,15 - 19 Years,211394,211067,209579,206749,204913,215234,222222,226520,232973,...,215200,198600,175800,159000,140000,124900,124700,131500,134300,135679
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
370,70 Years & Over,3093,2871,2697,2483,2295,2328,2197,2087,1948,...,400,400,400,400,400,400,400,400,400,305
371,75 Years & Over,1762,1630,1529,1423,1318,1337,1348,1362,1324,...,na,na,na,na,na,na,na,na,na,140
372,80 Years & Over,883,887,910,907,902,897,858,842,818,...,na,na,na,na,na,na,na,na,na,66
373,85 Years & Over,526,500,501,510,487,474,454,437,418,...,na,na,na,na,na,na,na,na,na,25


In [28]:
# get the data of interest 
target_ethnic_rows = ["male_malays",
                        "female_malays",
                        "male_chinese",
                        "female_chinese",
                        "male_indians",
                        "female_indians",
                        "male_others",
                        "female_others"]
rename_map = {
    'Total Male Malays': 'male_malays',
    'Total Female Malays': 'female_malays',
    'Total Male Chinese': 'male_chinese',
    'Total Female Chinese': 'female_chinese',
    'Total Male Indians': 'male_indians',
    'Total Female Indians': 'female_indians',
    'Other Ethnic Groups (Males)': 'male_others',
    'Other Ethnic Groups (Females)': 'female_others'
}

years_to_keep = [str(year) for year in range(2010, 2022)]
columns_to_keep = ['dataseries'] + [col for col in df_national.columns if col in years_to_keep]

df_national['dataseries'] = df_national['dataseries'].str.strip()
df_national['dataseries'] = df_national['dataseries'].replace(rename_map)

df_ethnic_national = df_national[
    df_national['dataseries'].isin(rename_map.values())
][columns_to_keep].copy()

# Reset index for a clean dataframe
df_ethnic_national = df_ethnic_national.reset_index(drop=True)

display(df_ethnic_national)

,dataseries,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012,2011,2010
0,male_malays,271743,271332,268933,266486,263912,261564,259108,257036,255176,253578,252227,250885
1,female_malays,272709,274166,271850,269338,266798,264324,261815,259621,257661,255941,254410,252983
2,male_chinese,1440854,1461344,1455708,1445375,1436284,1425244,1415303,1403927,1395183,1385606,1375665,1370083
3,female_chinese,1519281,1545425,1538000,1523906,1511968,1497928,1484704,1470453,1458568,1446378,1432589,1423897
4,male_indians,182594,185565,185343,184496,183818,182935,182304,181359,180889,180948,180548,180327
5,female_indians,172291,176709,177294,176032,175003,173941,172648,171662,170778,170077,168488,167792
6,male_others,57923,59315,59398,59481,59531,59783,59913,60088,60256,59914,59730,59838
7,female_others,69447,70354,69683,69169,68482,67840,66895,66593,66240,65763,65594,65916


In [29]:
# pivot the dataframe from wide to long format
df_ethnic_long = df_ethnic_national.melt(
    id_vars=['dataseries'], 
    var_name='Year', 
    value_name='NationalTotal'
)

# Clean up and reorder columns
df_ethnic_long['Year'] = df_ethnic_long['Year'].astype(int)
df_ethnic_long = df_ethnic_long[['dataseries', 'Year', 'NationalTotal']]
df_ethnic_long = df_ethnic_long.rename(columns = {"dataseries": "Ethnicity"})

display(df_ethnic_long)

,Ethnicity,Year,NationalTotal
0,male_malays,2021,271743
1,female_malays,2021,272709
2,male_chinese,2021,1440854
3,female_chinese,2021,1519281
4,male_indians,2021,182594
...,...,...,...
91,female_chinese,2010,1423897
92,male_indians,2010,180327
93,female_indians,2010,167792
94,male_others,2010,59838


In [30]:
def calculate_ethnicity_subzone_residents(df_ethnicity_density_ratio, df_ethnic_long):
    # 1. Normalize the Ethnicity Ratios
    # groupby ensures we normalize within each specific group/year combination
    df_ethnicity_density_ratio['Total_Ratio_Sum'] = df_ethnicity_density_ratio.groupby(['Year', 'Ethnicity'])['WeightingRatio'].transform('sum')
    
    # Calculate Normalized Share (Proportional weight of each subzone)
    df_ethnicity_density_ratio['Normalized_Share'] = df_ethnicity_density_ratio['WeightingRatio'] / df_ethnicity_density_ratio['Total_Ratio_Sum']
    
    # 2. Merge with National Ethnicity Totals
    # Ensure column names match: 'Year' and 'Ethnicity'
    df_final = pd.merge(
        df_ethnicity_density_ratio, 
        df_ethnic_long, 
        on=['Year', 'Ethnicity'], 
        how='inner'
    )
    
    # 3. Final Top-Down Allocation
    # Estimated_Residents = Normalized_Share * NationalTotal
    df_final['Estimated_Residents'] = (
        df_final['Normalized_Share'] * df_final['NationalTotal']
    )
    
    # 4. Cleanup and Type Casting
    df_final['Estimated_Residents'] = df_final['Estimated_Residents'].fillna(0).round(0).astype(int)
    
    # Reorder columns for clarity
    output_cols = ['subzone_n', 'pln_area_n', 'Year', 'Ethnicity', 'Estimated_Residents']
    return df_final[output_cols].sort_values(['subzone_n', 'Year', 'Ethnicity'])

# Generate the results
df_final_ethnicity_pop = calculate_ethnicity_subzone_residents(df_ethnicity_density_ratio, df_ethnic_long)

display(df_final_ethnicity_pop.head())

# Save the final ethnicity population data
save_path_ethnic = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/ethnic_density_ratios/final_subzone_ethnicity_population.csv")
df_final_ethnicity_pop.to_csv(save_path_ethnic, index=False)
    

,subzone_n,pln_area_n,Year,Ethnicity,Estimated_Residents
0,admiralty,sembawang,2010,female_chinese,4135
11,admiralty,sembawang,2010,female_indians,509
22,admiralty,sembawang,2010,female_malays,850
33,admiralty,sembawang,2010,female_others,208
44,admiralty,sembawang,2010,male_chinese,4076


In [31]:
# 1. Define the Year and specific Ethnicity/Sex group to check
check_year = 2014
check_ethnicity = 'male_chinese'  # Options: 'male_chinese', 'female_malays', etc.

# 2. Calculate the sum of allocated residents across all subzones
allocated_sum = df_final_ethnicity_pop[
    (df_final_ethnicity_pop['Year'] == check_year) & 
    (df_final_ethnicity_pop['Ethnicity'] == check_ethnicity)
]['Estimated_Residents'].sum()

# 3. Retrieve the actual national total for that year and group
actual_national = df_ethnic_long[
    (df_ethnic_long['Year'] == check_year) & 
    (df_ethnic_long['Ethnicity'] == check_ethnicity)
]['NationalTotal'].values[0]

# 4. Display the results
print(f"Validation for {check_ethnicity} in {check_year}:")
print(f"Total Allocated to Subzones: {allocated_sum:,}")
print(f"Actual National Total:       {actual_national:,}")
print(f"Difference (Rounding Error):  {actual_national - allocated_sum}")

Validation for male_chinese in 2014:
Total Allocated to Subzones: 1,403,926
Actual National Total:       1,403,927
Difference (Rounding Error):  1
